# Topic 07 — Solutions: DateTime & Time Series

*Reference solutions for the objective tasks. Try the practice first.* The Warm-Up concept questions, Interview Lens and Reflection have **no key** — by design.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 30)
RAW = '../datasets/raw/'
orders = pd.read_csv(RAW+'orders.csv', dtype={'order_id':str})
orders['channel'] = orders['channel'].str.strip().str.title()
items = pd.read_csv(RAW+'order_items.csv', dtype={'order_id':str,'product_id':str})
items['line_revenue'] = items['quantity']*items['unit_price']*(1-items['discount'])


### Warm-Up — NumPy recall (self-check)

In [ ]:
d = np.array(['2023-01-01','2023-02-01'], dtype='datetime64[D]')
gap = int((d[1]-d[0]) / np.timedelta64(1,'D'))
print(gap)

### Core lesson tasks

In [ ]:
orders['order_date'] = pd.to_datetime(orders['order_date'], format='mixed', dayfirst=True, errors='coerce')
print(orders['order_date'].notna().mean())

In [ ]:
oi = items.merge(orders[['order_id','order_date']], on='order_id', how='left')
ts = oi.dropna(subset=['order_date']).set_index('order_date').sort_index()
monthly = ts['line_revenue'].resample('ME').sum()
print(monthly.tail())

In [ ]:
mom = monthly.pct_change()
print(mom.idxmin(), mom.idxmax())

### Mixed review (earlier topics)

In [ ]:
oi['dow'] = oi['order_date'].dt.day_name()
print(oi.groupby('dow')['line_revenue'].sum())

In [ ]:
mk = pd.read_csv(RAW+'marketing_spend.csv', parse_dates=['date'])
daily = mk.groupby('date')['spend_gbp'].sum()
full = pd.date_range(daily.index.min(), daily.index.max(), freq='D')
print(len(full) - len(daily))

### Data detective

In [ ]:
oi = items.merge(orders[['order_id','order_date']], on='order_id', how='left')
ts = oi.dropna(subset=['order_date']).set_index('order_date').sort_index()
print(ts['line_revenue'].resample('ME').sum().describe())
# The 'dip' is seasonal (Nov/Dec peaks); once parsed, revenue is not structurally falling.

### Interview Lens — discussion points (not full answers)
- **Q22:** parse at load sets dtype once, avoids repeated conversions & id corruption; convert-after suits ad-hoc/inspection-first.
- **Q24:** fillna(0) claims 'nothing happened'; interpolation invents values. Reindex to a full range first; choose by what the series means.